### TwitchTracker Stream Data status

- Collected/loaded: yes
- Reviewed for usefulness: yes
- EDA needed: yes, light inspection and structure checks
- Cleaning/prep completed: yes
- Saved as separate processed file: yes, output saved as `data/processed/twitch_twitchtracker_stream_data_processed.csv`
- Ready for SQL/Tableau: yes, with one limitation
- Role: main Twitch stream-level dataset for event coverage, channel activity, and stream performance analysis
- Key limitation: `views_gained` is only available through 2021 and is missing from 2022 onward, so it should not be used for full-period comparisons

### Notes / important decisions

- One row from 2023 was misparsed at import because the stream title contained a comma. That row was corrected manually in the notebook.
- One duration value appeared as `7h` instead of `7h0m`. It was standardized before creating `duration_minutes`.
- `views_gained` is only available through 2021 and is fully missing from 2022 onward. This column can be kept, but it should only be used for partial-period analysis.
- Multiple Twitch channels are included in this dataset (`classictetris`, `classictetris2`, etc.), so `channel_name` was extracted from the URL and kept as its own field.
- Stream titles are not unique. The same title can appear multiple times on the same date and channel, so titles should not be used as unique identifiers.
- `twitchtracker_url` and extracted `source_stream_id` are unique for each row and should be treated as the source-level identifiers.
- `hours_watched` is an audience watch-time metric, not stream duration.

In [59]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

In [2]:
file_path = "../data/raw/ctwc_twitchtracker_stream_data.csv"

df = pd.read_csv(file_path)
df.head()

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
0,2017,2017-10-22,2017 Classic Tetris World Championship,7h19m,1121,1790,8201,358,11362.0,https://twitchtracker.com/classictetris/streams/26546739632
1,2018,2018-10-21,2018 Classic Tetris World Championship Day 3,6h56m,6926,24706,48020,2058,213389.0,https://twitchtracker.com/classictetris/streams/30874738064
2,2019,2019-10-20,CTWC 2019 - Top 32 and 16@10AM PST - Top 8 at 2PM PST - Finals at 5PM PST,8h3m,9259,17554,74534,2954,337915.0,https://twitchtracker.com/classictetris/streams/36024688528
3,2020,2020-10-31,2020 CTWC - GROUP B with Chris Tang and Vandweller,8h40m,3239,8074,28071,509,141784.0,https://twitchtracker.com/classictetris/streams/40295577518
4,2020,2020-11-01,CTWC 2020 - Group E Bracket /w Vandweller and Chris Tang!,7h36m,4362,17621,33151,711,211786.0,https://twitchtracker.com/classictetris/streams/40309358270


In [3]:
df.shape

(130, 10)

In [5]:
df.columns

Index(['year', 'stream_date', 'stream_title', 'duration', 'avg_viewers', 'peak_viewers', 'hours_watched', 'followers_gained', 'views_gained', 'twitchtracker_url'], dtype='str')

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year               130 non-null    int64  
 1   stream_date        130 non-null    str    
 2   stream_title       130 non-null    str    
 3   duration           130 non-null    str    
 4   avg_viewers        130 non-null    str    
 5   peak_viewers       130 non-null    int64  
 6   hours_watched      130 non-null    int64  
 7   followers_gained   130 non-null    int64  
 8   views_gained       63 non-null     float64
 9   twitchtracker_url  130 non-null    str    
dtypes: float64(1), int64(4), str(5)
memory usage: 10.3 KB


In [7]:
df.isna().sum()

year                  0
stream_date           0
stream_title          0
duration              0
avg_viewers           0
peak_viewers          0
hours_watched         0
followers_gained      0
views_gained         67
twitchtracker_url     0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df.sample(10, random_state=42)

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
55,2019,2019-10-20,2019 CTWC Round 1 - Side Station,3h16m,1551,2365,5066,433,4540.0,https://twitchtracker.com/classictetris2/streams/36024742640
40,2020,2020-10-13,2020 CTWC Qualifying Spotlight,2h26m,243,280,591,30,833.0,https://twitchtracker.com/classictetris/streams/40073424366
19,2022,2022-10-16,CTWC 2022 Day 3 - Gold Bracket Finals,7h42m,3859,6232,29714,466,NaN,https://twitchtracker.com/classictetris/streams/41397993435
31,2025,2025-06-08,CLASSIC TETRIS WORLD CHAMPIONSHIP | !MATCHERINO !QUAL !BRACKET !SU2C !DONATE,6h20m,2483,3329,15725,269,NaN,https://twitchtracker.com/classictetris/streams/322392046194
115,2025,2025-06-08,[FR] Top 16 du CTWC 2025 en Français ! Commenté par @GunterTetris,7h46m,12,44,93,9,NaN,https://twitchtracker.com/classictetris5/streams/321419888505
56,2020,2020-10-31,CTWC 2020 Group F Top 64,7h23m,931,1375,6873,526,4293.0,https://twitchtracker.com/classictetris2/streams/39626849917
69,2022,2022-10-16,CTWC 2022 Day 3 - Gold Bracket Finals,2h43m,810,1546,2200,75,NaN,https://twitchtracker.com/classictetris2/streams/41397896427
105,2024,2024-06-08,Classic Tetris World Championship XV - Bronze Bracket,8h12m,60,95,492,71,NaN,https://twitchtracker.com/classictetris4/streams/44319402475
81,2025,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | QUALS & BRACKET PLAY | !MATCHERINO !QUAL !BRACKET !DONATE,4h15m,188,332,799,30,NaN,https://twitchtracker.com/classictetris2/streams/322247074172
26,2024,2024-06-09,Classic Tetris World Championship XV - Day 3 Finals,9h40m,3897,6068,37671,1147,NaN,https://twitchtracker.com/classictetris/streams/44319075547


In [10]:
df["avg_viewers"].sample(20, random_state=42)

55     1551
40      243
19     3859
31     2483
115      12
56      931
69      810
105      60
81      188
26     3897
95      109
27      612
64     1423
4      4362
97      134
100      65
36      474
80      335
93       17
84       29
Name: avg_viewers, dtype: str

In [11]:
df["avg_viewers"].map(type).value_counts()

avg_viewers
<class 'str'>    130
Name: count, dtype: int64

In [12]:
df["avg_viewers"].str.contains(r"[^0-9]", na=False).sum()

np.int64(1)

In [13]:
df.loc[df["avg_viewers"].str.contains(r"[^0-9]", na=False), "avg_viewers"].drop_duplicates().sort_values()

21    7h27m
Name: avg_viewers, dtype: str

In [14]:
df.loc[df["avg_viewers"] == "7h27m"]

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
21,2023,2023-10-14,CTWC 2023 Day 2: Final Quals,Gold Bracket | !qual !squad !jersey !matcherino,7h27m,1431,2228,10660,105.0,https://twitchtracker.com/classictetris/streams/49489971261


In [15]:
df[df["year"] == 2023].head(10)

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
20,2023,2023-10-13,CTWC 2023 Day 1: Qualifying | !qual !squad !jersey !matcherino,9h26m,869,1109,8197,94,NaN,https://twitchtracker.com/classictetris/streams/49484154941
21,2023,2023-10-14,CTWC 2023 Day 2: Final Quals,Gold Bracket | !qual !squad !jersey !matcherino,7h27m,1431,2228,10660,105.0,https://twitchtracker.com/classictetris/streams/49489971261
22,2023,2023-10-15,CTWC 2023 Day 2: Gold Bracket Top 32 | !qual !squad !jersey !matcherino,4h50m,1238,1637,5983,55,NaN,https://twitchtracker.com/classictetris/streams/49493489101
23,2023,2023-10-15,CTWC 2023 Day 3: Gold Bracket FINAL | !bracket !squad !jersey !matcherino,8h18m,3048,4471,25298,224,NaN,https://twitchtracker.com/classictetris/streams/49497856333
72,2023,2023-10-15,CTWC 2023 Day 3: Gold Bracket Top 8 EN ESPAÑOL | !jersey !bracket !matcherino,5h40m,45,74,255,10,NaN,https://twitchtracker.com/classictetris2/streams/42916729323
73,2023,2023-10-14,"CTWC 2023 Day 2: Final Quals, Gold Bracket | !qual !jersey !matcherino",9h53m,375,729,3706,114,NaN,https://twitchtracker.com/classictetris2/streams/42910549195
74,2023,2023-10-13,CTWC 2023 Day 1: Qualifying | !qual !jersey !matcherino,9h36m,176,370,1689,82,NaN,https://twitchtracker.com/classictetris2/streams/42906022571
90,2023,2023-10-15,[日本語] CTWC 2023 Japanese Commentary | !qual !jersey !matcherino,8h6m,43,70,348,25,NaN,https://twitchtracker.com/classictetris3/streams/40719793655
91,2023,2023-10-14,CTWC 2023 Day 1: Qualifying | !qual !jersey !matcherino,9h31m,98,723,932,148,NaN,https://twitchtracker.com/classictetris3/streams/42910552587
92,2023,2023-10-13,CTWC 2023 Day 1: Qualifying | !qual !jersey !matcherino,9h6m,72,112,655,165,NaN,https://twitchtracker.com/classictetris3/streams/42906029387


In [16]:
import csv
raw_check = pd.read_csv(
    "../data/raw/ctwc_twitchtracker_stream_data.csv",
    sep=",",
    engine="python",
    quoting=csv.QUOTE_MINIMAL,
    on_bad_lines="warn"
)
raw_check.loc[raw_check["year"] == 2023].head(10)

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
20,2023,2023-10-13,CTWC 2023 Day 1: Qualifying | !qual !squad !jersey !matcherino,9h26m,869,1109,8197,94,NaN,https://twitchtracker.com/classictetris/streams/49484154941
21,2023,2023-10-14,CTWC 2023 Day 2: Final Quals,Gold Bracket | !qual !squad !jersey !matcherino,7h27m,1431,2228,10660,105.0,https://twitchtracker.com/classictetris/streams/49489971261
22,2023,2023-10-15,CTWC 2023 Day 2: Gold Bracket Top 32 | !qual !squad !jersey !matcherino,4h50m,1238,1637,5983,55,NaN,https://twitchtracker.com/classictetris/streams/49493489101
23,2023,2023-10-15,CTWC 2023 Day 3: Gold Bracket FINAL | !bracket !squad !jersey !matcherino,8h18m,3048,4471,25298,224,NaN,https://twitchtracker.com/classictetris/streams/49497856333
72,2023,2023-10-15,CTWC 2023 Day 3: Gold Bracket Top 8 EN ESPAÑOL | !jersey !bracket !matcherino,5h40m,45,74,255,10,NaN,https://twitchtracker.com/classictetris2/streams/42916729323
73,2023,2023-10-14,"CTWC 2023 Day 2: Final Quals, Gold Bracket | !qual !jersey !matcherino",9h53m,375,729,3706,114,NaN,https://twitchtracker.com/classictetris2/streams/42910549195
74,2023,2023-10-13,CTWC 2023 Day 1: Qualifying | !qual !jersey !matcherino,9h36m,176,370,1689,82,NaN,https://twitchtracker.com/classictetris2/streams/42906022571
90,2023,2023-10-15,[日本語] CTWC 2023 Japanese Commentary | !qual !jersey !matcherino,8h6m,43,70,348,25,NaN,https://twitchtracker.com/classictetris3/streams/40719793655
91,2023,2023-10-14,CTWC 2023 Day 1: Qualifying | !qual !jersey !matcherino,9h31m,98,723,932,148,NaN,https://twitchtracker.com/classictetris3/streams/42910552587
92,2023,2023-10-13,CTWC 2023 Day 1: Qualifying | !qual !jersey !matcherino,9h6m,72,112,655,165,NaN,https://twitchtracker.com/classictetris3/streams/42906029387


In [17]:
broken_rows = df[df["avg_viewers"].str.contains(r"^\d+h\d+m$", na=False)].copy()
broken_rows

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
21,2023,2023-10-14,CTWC 2023 Day 2: Final Quals,Gold Bracket | !qual !squad !jersey !matcherino,7h27m,1431,2228,10660,105.0,https://twitchtracker.com/classictetris/streams/49489971261


In [19]:
mask = df["avg_viewers"].str.contains(r"^\d+h\d+m$", na=False)

cols_to_fix = [
    "stream_title",
    "duration",
    "avg_viewers",
    "peak_viewers",
    "hours_watched",
    "followers_gained",
    "views_gained"
]

df[cols_to_fix] = df[cols_to_fix].astype("object")

old_title = df.loc[mask, "stream_title"].copy()
old_duration = df.loc[mask, "duration"].copy()
old_avg = df.loc[mask, "avg_viewers"].copy()
old_peak = df.loc[mask, "peak_viewers"].copy()
old_hours = df.loc[mask, "hours_watched"].copy()
old_followers = df.loc[mask, "followers_gained"].copy()
old_views = df.loc[mask, "views_gained"].copy()

df.loc[mask, "stream_title"] = old_title + ", " + old_duration
df.loc[mask, "duration"] = old_avg
df.loc[mask, "avg_viewers"] = old_peak
df.loc[mask, "peak_viewers"] = old_hours
df.loc[mask, "hours_watched"] = old_followers
df.loc[mask, "followers_gained"] = old_views
df.loc[mask, "views_gained"] = np.nan

In [20]:
df.loc[mask]

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
21,2023,2023-10-14,"CTWC 2023 Day 2: Final Quals, Gold Bracket | !qual !squad !jersey !matcherino, 7h27m",7h27m,1431,2228,10660,105.0,NaN,https://twitchtracker.com/classictetris/streams/49489971261


In [21]:
df["avg_viewers"].astype(str).str.contains(r"[^0-9]", na=False).sum()

np.int64(0)

In [22]:
df.loc[21, "stream_title"] = "CTWC 2023 Day 2: Final Quals, Gold Bracket | !qual !squad !jersey !matcherino"
df.loc[21, "duration"] = "7h27m"
df.loc[21, "avg_viewers"] = 1431
df.loc[21, "peak_viewers"] = 2228
df.loc[21, "hours_watched"] = 10660
df.loc[21, "followers_gained"] = 105
df.loc[21, "views_gained"] = np.nan

In [23]:
df.loc[[21]]

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
21,2023,2023-10-14,"CTWC 2023 Day 2: Final Quals, Gold Bracket | !qual !squad !jersey !matcherino",7h27m,1431,2228,10660,105,NaN,https://twitchtracker.com/classictetris/streams/49489971261


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   year               130 non-null    int64 
 1   stream_date        130 non-null    str   
 2   stream_title       130 non-null    object
 3   duration           130 non-null    object
 4   avg_viewers        130 non-null    object
 5   peak_viewers       130 non-null    object
 6   hours_watched      130 non-null    object
 7   followers_gained   130 non-null    object
 8   views_gained       62 non-null     object
 9   twitchtracker_url  130 non-null    str   
dtypes: int64(1), object(7), str(2)
memory usage: 10.3+ KB


In [25]:
df["duration"].sample(20, random_state=42)

55      3h16m
40      2h26m
19      7h42m
31      6h20m
115     7h46m
56      7h23m
69      2h43m
105     8h12m
81      4h15m
26      9h40m
95      7h15m
27     10h21m
64      7h56m
4       7h36m
97     11h36m
100      8h5m
36      8h55m
80       7h0m
93      6h55m
84      7h46m
Name: duration, dtype: object

In [26]:
df["stream_date"] = pd.to_datetime(df["stream_date"], errors="coerce")

duration_parts = df["duration"].astype(str).str.extract(r"(?P<hours>\d+)h(?P<minutes>\d+)m")
df["duration_minutes"] = (
    pd.to_numeric(duration_parts["hours"], errors="coerce") * 60
    + pd.to_numeric(duration_parts["minutes"], errors="coerce")
)

numeric_cols = [
    "avg_viewers",
    "peak_viewers",
    "hours_watched",
    "followers_gained",
    "views_gained"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   year               130 non-null    int64         
 1   stream_date        130 non-null    datetime64[us]
 2   stream_title       130 non-null    object        
 3   duration           130 non-null    object        
 4   avg_viewers        130 non-null    int64         
 5   peak_viewers       130 non-null    int64         
 6   hours_watched      130 non-null    int64         
 7   followers_gained   130 non-null    int64         
 8   views_gained       62 non-null     float64       
 9   twitchtracker_url  130 non-null    str           
 10  duration_minutes   129 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(5), object(2), str(1)
memory usage: 11.3+ KB


In [28]:
df.isna().sum()

year                  0
stream_date           0
stream_title          0
duration              0
avg_viewers           0
peak_viewers          0
hours_watched         0
followers_gained      0
views_gained         68
twitchtracker_url     0
duration_minutes      1
dtype: int64

In [31]:
df.loc[df["duration_minutes"].isna(), ["year", "stream_date", "stream_title", "duration"]]

,year,stream_date,stream_title,duration
29,2025,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP | !MATCHERINO !QUAL !BRACKET !SU2C !DONATE,7h


In [32]:
df.loc[df["duration"] == "7h", "duration"] = "7h0m"

In [33]:
duration_parts = df["duration"].astype(str).str.extract(r"(?P<hours>\d+)h(?P<minutes>\d+)m")
df["duration_minutes"] = (
    pd.to_numeric(duration_parts["hours"], errors="coerce") * 60
    + pd.to_numeric(duration_parts["minutes"], errors="coerce")
)

In [34]:
df.loc[df["year"] == 2025, ["stream_date", "stream_title", "duration", "duration_minutes"]].sort_values("stream_date")

,stream_date,stream_title,duration,duration_minutes
27,2025-06-06,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | QUALIFYING | !MATCHERINO !QUAL !SU2C !DONATE,10h21m,621
109,2025-06-06,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | QUALIFYING,7h46m,466
98,2025-06-06,2025 Classic Tetris World Championship | Qualifying | Day 1 | !qual !matcherino,8h28m,508
83,2025-06-06,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | QUALIFYING | !MATCHERINO !QUAL !DONATE,7h40m,460
80,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP | !MATCHERINO !QUAL !BRACKET !SU2C !DONATE,7h0m,420
81,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | QUALS & BRACKET PLAY | !MATCHERINO !QUAL !BRACKET !DONATE,4h15m,255
82,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | TGM4 JPN Commentary | !MATCHERINO !QUAL !DONATE,2h9m,129
97,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | SILVER BRACKET | !MATCHERINO !BRACKET !QUAL !DONATE,11h36m,696
29,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP | !MATCHERINO !QUAL !BRACKET !SU2C !DONATE,7h0m,420
108,2025-06-07,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | BRONZE BRACKET | !MATCHERINO !BRACKET !QUAL !DONATE,6h34m,394


In [35]:
df["duration_minutes"].isna().sum()

np.int64(0)

In [29]:
df[["year", "stream_date", "views_gained"]].sort_values("stream_date")

,year,stream_date,views_gained
52,2017,2017-10-21,4292.0
0,2017,2017-10-22,11362.0
51,2018,2018-10-20,1169.0
50,2018,2018-10-20,15216.0
1,2018,2018-10-21,213389.0
53,2019,2019-10-18,1400.0
48,2019,2019-10-18,308406.0
49,2019,2019-10-19,89124.0
54,2019,2019-10-19,3281.0
55,2019,2019-10-20,4540.0


In [30]:
df.groupby("year")["views_gained"].apply(lambda x: x.notna().sum())

year
2017     2
2018     3
2019     6
2020    20
2021    31
2022     0
2023     0
2024     0
2025     0
Name: views_gained, dtype: int64

In [36]:
df.head()

,year,stream_date,stream_title,duration,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url,duration_minutes
0,2017,2017-10-22,2017 Classic Tetris World Championship,7h19m,1121,1790,8201,358,11362.0,https://twitchtracker.com/classictetris/streams/26546739632,439
1,2018,2018-10-21,2018 Classic Tetris World Championship Day 3,6h56m,6926,24706,48020,2058,213389.0,https://twitchtracker.com/classictetris/streams/30874738064,416
2,2019,2019-10-20,CTWC 2019 - Top 32 and 16@10AM PST - Top 8 at 2PM PST - Finals at 5PM PST,8h3m,9259,17554,74534,2954,337915.0,https://twitchtracker.com/classictetris/streams/36024688528,483
3,2020,2020-10-31,2020 CTWC - GROUP B with Chris Tang and Vandweller,8h40m,3239,8074,28071,509,141784.0,https://twitchtracker.com/classictetris/streams/40295577518,520
4,2020,2020-11-01,CTWC 2020 - Group E Bracket /w Vandweller and Chris Tang!,7h36m,4362,17621,33151,711,211786.0,https://twitchtracker.com/classictetris/streams/40309358270,456


In [37]:
df["channel_name"] = df["twitchtracker_url"].str.extract(r"twitchtracker\.com/([^/]+)/")

In [38]:
df[["twitchtracker_url", "channel_name"]].head()

,twitchtracker_url,channel_name
0,https://twitchtracker.com/classictetris/streams/26546739632,classictetris
1,https://twitchtracker.com/classictetris/streams/30874738064,classictetris
2,https://twitchtracker.com/classictetris/streams/36024688528,classictetris
3,https://twitchtracker.com/classictetris/streams/40295577518,classictetris
4,https://twitchtracker.com/classictetris/streams/40309358270,classictetris


In [39]:
df["channel_name"].value_counts()

channel_name
classictetris     53
classictetris2    35
classictetris3    15
classictetris4    11
classictetris5     6
classictetris7     5
classictetris8     5
Name: count, dtype: int64

In [40]:
df["twitchtracker_url"].nunique(), len(df)

(130, 130)

In [41]:
df.duplicated(subset=["stream_date", "stream_title", "channel_name"]).sum()

np.int64(5)

In [42]:
dup_mask = df.duplicated(subset=["stream_date", "stream_title", "channel_name"], keep=False)

df.loc[dup_mask, [
    "year",
    "stream_date",
    "channel_name",
    "stream_title",
    "duration",
    "duration_minutes",
    "avg_viewers",
    "peak_viewers",
    "hours_watched",
    "followers_gained",
    "views_gained",
    "twitchtracker_url"
]].sort_values(["stream_date", "channel_name", "stream_title"])

,year,stream_date,channel_name,stream_title,duration,duration_minutes,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,twitchtracker_url
33,2021,2021-09-30,classictetris,CTWC 2021 || Qualification Showcase || !qual !merch,2h13m,133,272,391,602,-2,444.0,https://twitchtracker.com/classictetris/streams/43894641981
34,2021,2021-09-30,classictetris,CTWC 2021 || Qualification Showcase || !qual !merch,15h5m,905,462,644,6968,29,6484.0,https://twitchtracker.com/classictetris/streams/43895833869
35,2021,2021-09-30,classictetris,CTWC 2021 || Qualification Showcase || !qual !merch,1h50m,110,147,181,269,3,272.0,https://twitchtracker.com/classictetris/streams/43892927741
125,2021,2021-10-02,classictetris2,CTWC 2021 || Qualification Showcase || !qual !merch,4h37m,277,114,154,526,32,538.0,https://twitchtracker.com/classictetris2/streams/43923062829
126,2021,2021-10-02,classictetris2,CTWC 2021 || Qualification Showcase || !qual !merch,1h1m,61,62,72,63,3,47.0,https://twitchtracker.com/classictetris2/streams/43921483869
127,2021,2021-10-02,classictetris2,CTWC 2021 || Qualification Showcase || !qual !merch,5h44m,344,170,288,974,50,746.0,https://twitchtracker.com/classictetris2/streams/43912728957
66,2021,2021-10-03,classictetris2,CTWC 2021 || Qualification Showcase || !qual !merch,10h8m,608,214,343,2168,82,2119.0,https://twitchtracker.com/classictetris2/streams/43936783597
67,2021,2021-10-03,classictetris2,CTWC 2021 || Qualification Showcase || !qual !merch,4h38m,278,154,209,713,20,563.0,https://twitchtracker.com/classictetris2/streams/43926722685


In [43]:
df["source_stream_id"] = df["twitchtracker_url"].str.extract(r"/streams/(\d+)$")

In [44]:
df[["twitchtracker_url", "source_stream_id"]].head()

,twitchtracker_url,source_stream_id
0,https://twitchtracker.com/classictetris/streams/26546739632,26546739632
1,https://twitchtracker.com/classictetris/streams/30874738064,30874738064
2,https://twitchtracker.com/classictetris/streams/36024688528,36024688528
3,https://twitchtracker.com/classictetris/streams/40295577518,40295577518
4,https://twitchtracker.com/classictetris/streams/40309358270,40309358270


In [45]:
df["source_stream_id"].duplicated().sum(), df["source_stream_id"].isna().sum()

(np.int64(0), np.int64(0))

In [47]:
final_cols = [
    "year",
    "stream_date",
    "channel_name",
    "stream_title",
    "duration",
    "duration_minutes",
    "avg_viewers",
    "peak_viewers",
    "hours_watched",
    "followers_gained",
    "views_gained",
    "source_stream_id",
    "twitchtracker_url"
]

df_clean = df[final_cols].copy()

In [48]:
df_clean["source_stream_id"] = pd.to_numeric(df_clean["source_stream_id"], errors="coerce")

df_clean = df_clean.sort_values(
    by=["year", "stream_date", "channel_name", "source_stream_id"],
    ascending=[True, True, True, True]
).reset_index(drop=True)

In [49]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   year               130 non-null    int64         
 1   stream_date        130 non-null    datetime64[us]
 2   channel_name       130 non-null    str           
 3   stream_title       130 non-null    object        
 4   duration           130 non-null    object        
 5   duration_minutes   130 non-null    int64         
 6   avg_viewers        130 non-null    int64         
 7   peak_viewers       130 non-null    int64         
 8   hours_watched      130 non-null    int64         
 9   followers_gained   130 non-null    int64         
 10  views_gained       62 non-null     float64       
 11  source_stream_id   130 non-null    int64         
 12  twitchtracker_url  130 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(7), object(2), str(2)
memory usage: 

In [50]:
df_clean[["year", "stream_date", "channel_name", "stream_title"]].head(20)

,year,stream_date,channel_name,stream_title
0,2017,2017-10-21,classictetris,2017 Classic Tetris World Championship Qualifying Rounds
1,2017,2017-10-22,classictetris,2017 Classic Tetris World Championship
2,2018,2018-10-20,classictetris,2018 Classic Tetris World Championship Opening Day
3,2018,2018-10-20,classictetris,2018 Classic Tetris World Championship Day 2
4,2018,2018-10-21,classictetris,2018 Classic Tetris World Championship Day 3
5,2019,2019-10-18,classictetris,"CTWC 2019 - Tetris Speedrun @ 3pm, Dr. Mario @ 4pm"
6,2019,2019-10-18,classictetris2,2019 CTWC Mindmeld
7,2019,2019-10-19,classictetris,"CTWC 2019 - Tetris Speedrun @ 3pm, Dr. Mario @ 4pm"
8,2019,2019-10-19,classictetris2,2019 CTWC Side Station Qualifying
9,2019,2019-10-20,classictetris,CTWC 2019 - Top 32 and 16@10AM PST - Top 8 at 2PM PST - Finals at 5PM PST


In [51]:
df_clean[["year", "stream_date", "channel_name", "stream_title"]].tail(20)

,year,stream_date,channel_name,stream_title
110,2025,2025-06-06,classictetris,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | QUALIFYING | !MATCHERINO !QUAL !SU2C !DONATE
111,2025,2025-06-06,classictetris2,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | QUALIFYING | !MATCHERINO !QUAL !DONATE
112,2025,2025-06-06,classictetris3,2025 Classic Tetris World Championship | Qualifying | Day 1 | !qual !matcherino
113,2025,2025-06-06,classictetris4,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | QUALIFYING
114,2025,2025-06-07,classictetris,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | QUALS & BRACKET PLAY | !MATCHERINO !BRACKET !QUAL !SU2C !DONATE
115,2025,2025-06-07,classictetris,CLASSIC TETRIS WORLD CHAMPIONSHIP | !MATCHERINO !QUAL !BRACKET !SU2C !DONATE
116,2025,2025-06-07,classictetris2,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | QUALS & BRACKET PLAY | !MATCHERINO !QUAL !BRACKET !DONATE
117,2025,2025-06-07,classictetris2,CLASSIC TETRIS WORLD CHAMPIONSHIP | !MATCHERINO !QUAL !BRACKET !SU2C !DONATE
118,2025,2025-06-07,classictetris2,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 1 | TGM4 JPN Commentary | !MATCHERINO !QUAL !DONATE
119,2025,2025-06-07,classictetris3,CLASSIC TETRIS WORLD CHAMPIONSHIP DAY 2 | SILVER BRACKET | !MATCHERINO !BRACKET !QUAL !DONATE


In [58]:
output_path = Path("../data/processed/twitch_twitchtracker_stream_data_processed.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_path, index=False)

pd.read_csv(output_path).head()

,year,stream_date,channel_name,stream_title,duration,duration_minutes,avg_viewers,peak_viewers,hours_watched,followers_gained,views_gained,source_stream_id,twitchtracker_url
0,2017,2017-10-21,classictetris,2017 Classic Tetris World Championship Qualifying Rounds,6h17m,377,330,411,2073,121,4292.0,26538549600,https://twitchtracker.com/classictetris/streams/26538549600
1,2017,2017-10-22,classictetris,2017 Classic Tetris World Championship,7h19m,439,1121,1790,8201,358,11362.0,26546739632,https://twitchtracker.com/classictetris/streams/26546739632
2,2018,2018-10-20,classictetris,2018 Classic Tetris World Championship Opening Day,3h19m,199,254,425,842,54,1169.0,30847269424,https://twitchtracker.com/classictetris/streams/30847269424
3,2018,2018-10-20,classictetris,2018 Classic Tetris World Championship Day 2,7h35m,455,1052,1589,7977,347,15216.0,30858710656,https://twitchtracker.com/classictetris/streams/30858710656
4,2018,2018-10-21,classictetris,2018 Classic Tetris World Championship Day 3,6h56m,416,6926,24706,48020,2058,213389.0,30874738064,https://twitchtracker.com/classictetris/streams/30874738064
